In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import libpysal
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from spopt.region import Skater, WardSpatial
from pathlib import Path
import re

In [ ]:
gdfs_reg = ["south", "chicago", "ceara_zika"]

dataset_variables = {
    "south"      : ["HR90","UE90","DV90","FP89","BLK90"],
    "chicago"    : ['poverty', 'crowded', 'dependency', 'without_hs', 'unemployed','income_pc', 'harship_in', 'perc_crimes', 'perc_theft'],
    "ceara_zika" : ['mobility', 'environ','housing', 'sanitation', 'infra'],
}

dataset_methods = {
    "south"      : ["random"],
    "chicago"    : ["random"],
    "ceara_zika" : ["random"],
}


c_runs      = range(10)   # corruption runs 0-9
scaler = StandardScaler()
GRAPH_NAME_RE = re.compile(r"g_missing(\d+)_run(\d+)\.parquet$")

In [ ]:
def parse_missing_edges(graph_path):
    m = GRAPH_NAME_RE.search(str(graph_path))
    if not m:
        raise ValueError(f"could not parse missing-edge count from: {graph_path}")
    return int(m.group(1))
 
 
def get_available_intensities(gdf_name, method, c_run, subfolder):
    directory = Path(f"graphs/real/{gdf_name}/{subfolder}/{method}")
    found = []
    for f in directory.glob("g_missing*_run*.parquet"):
        m = GRAPH_NAME_RE.match(f.name)
        if m and int(m.group(2)) == int(c_run):
            found.append(int(m.group(1)))
    return sorted(found)
 
 
def select_sample_graphs(available_intensities, n_sample=20):
    available_intensities = sorted(available_intensities)
    max_idx = len(available_intensities) - 1
    idxs = np.linspace(0, max_idx, n_sample, dtype=int)
    #idxs = idxs[1:]
    idxs = sorted(set(idxs))  # dedup: linspace can repeat if max_idx < n_sample - 1
    return [available_intensities[i] for i in idxs]

def get_available_c_runs(gdf_name, method, subfolder="one_component"):
    directory = Path(f"graphs/real/{gdf_name}/{subfolder}/{method}")
    found = set()
    for f in directory.glob("g_missing*_run*.parquet"):
        m = GRAPH_NAME_RE.match(f.name)
        if m:
            found.add(m.group(2))  # keep as string, preserves "05" vs 5
    return sorted(found, key=int)

In [ ]:
def _build_result_df_real(gdf_geometry, method, missing_edges, c_run, ward_labels, skater_labels):
    """Assembles a tidy long-format DataFrame slice for one real-data task combination."""
    n = len(gdf_geometry)
    return pd.DataFrame({
        "obs_id"      : gdf_geometry.index.values,
        "method"      : [method]    * n,
        "missing_edges"   : np.full(n, missing_edges, dtype=np.int32),
        "c_run"       : np.full(n, c_run,     dtype=np.int8),
        "ward_label"  : ward_labels,
        "skater_label": skater_labels,
    })


In [ ]:
def _process_regionalization_real(gdf_name, missing_edges, c_run, method, gdf_geometry, gdf_attributes, cluster_variables):
    """
    Worker function for real datasets.
    No z_run dimension — variables are fixed per dataset.
    """
    n = len(gdf_geometry)

    ward_labels   = np.full(n, -1, dtype=np.int8)
    skater_labels = np.full(n, -1, dtype=np.int8)

    var_scaled = StandardScaler().fit_transform(gdf_attributes[cluster_variables].values)
    graph_path = f"graphs/real/{gdf_name}/one_component/{method}/g_missing{missing_edges}_run{c_run}.parquet"
    missing_edges = parse_missing_edges(graph_path)
    try:
        w = libpysal.graph.read_parquet(graph_path).to_W()
    except FileNotFoundError:
        return _build_result_df_real(gdf_geometry, method, missing_edges, c_run, ward_labels, skater_labels)

    # Build local GDF once — shared by both algorithms
    local_gdf = gpd.GeoDataFrame(geometry=gdf_geometry)
    temp_cols = [f"v_{i}" for i in range(len(cluster_variables))]
    local_gdf[temp_cols] = var_scaled

    # SCHC
    try:
        model_ward = WardSpatial(
            gdf=local_gdf,
            w=w,
            attrs_name=temp_cols,
            n_clusters=5
        )
        model_ward.solve()
        ward_labels = model_ward.labels_.astype(np.int8)
    except Exception:
        pass

    # SKATER
    try:
        model_skater = Skater(
            gdf=local_gdf,
            w=w,
            attrs_name=temp_cols,
            n_clusters=5,
            floor=1,
            islands="ignore"
        )
        model_skater.solve()
        skater_labels = model_skater.labels_.astype(np.int8)
    except Exception:
        pass

    return _build_result_df_real(gdf_geometry, method,missing_edges, c_run, ward_labels, skater_labels)

In [ ]:
N_SAMPLE = 20
for gdf_name in gdfs_reg:
    print(f"Executing regionalization for {gdf_name}...")

    gdf = gpd.read_parquet(f'data/real/{gdf_name}.parquet')

    gdf_geometry      = gdf.geometry.copy()
    cluster_variables = dataset_variables[gdf_name]
    methods           = dataset_methods[gdf_name]
    gdf_attributes    = gdf[cluster_variables].copy()

    task_combinations = []
    for method in methods:
        #print (method)
        for c_run in c_runs:
            available = get_available_intensities(gdf_name,method, c_run, subfolder="one_component")
            #print (available)
            sampled = select_sample_graphs(available, n_sample=N_SAMPLE)
            for intensity in sampled:
                task_combinations.append((gdf_name,intensity, c_run, method))
    

    results = Parallel(n_jobs=-1)(
        delayed(_process_regionalization_real)(
            gdf_name, missing_edges, c_run, method, gdf_geometry, gdf_attributes, cluster_variables
        )
        for gdf_name,missing_edges, c_run, method in task_combinations
    )

    res_df = pd.concat(results, ignore_index=True)

    output_path = f"results/regionalization/real/reg_{gdf_name}.parquet"
    res_df.to_parquet(output_path, index=False)
    print(f"Saved: {output_path}  |  shape: {res_df.shape}  |  tasks: {len(task_combinations)}\n")

In [ ]:
N_SAMPLE = 50
for gdf_name in gdfs_reg:
    print(f"Executing regionalization for {gdf_name}...")

    gdf = gpd.read_parquet(f'data/real/{gdf_name}.parquet')

    gdf_geometry      = gdf.geometry.copy()
    cluster_variables = dataset_variables[gdf_name]
    methods           = dataset_methods[gdf_name]
    gdf_attributes    = gdf[cluster_variables].copy()

    task_combinations = []
    for method in methods:
        #print (method)
        for c_run in c_runs:
            available = numbers = list(range(1, 51))
            #print (available)
            sampled = select_sample_graphs(available, n_sample=N_SAMPLE)
            for intensity in sampled:
                task_combinations.append((gdf_name,intensity, c_run, method))
    

    results = Parallel(n_jobs=-1)(
        delayed(_process_regionalization_real)(
            gdf_name, missing_edges, c_run, method, gdf_geometry, gdf_attributes, cluster_variables
        )
        for gdf_name,missing_edges, c_run, method in task_combinations
    )

    res_df = pd.concat(results, ignore_index=True)

    output_path = f"results/regionalization/real/extra/reg_{gdf_name}.parquet"
    res_df.to_parquet(output_path, index=False)
    print(f"Saved: {output_path}  |  shape: {res_df.shape}  |  tasks: {len(task_combinations)}\n")

In [ ]:
for gdf_name in ["south"]:
    print(f"Executing regionalization for {gdf_name}...")

    gdf = gpd.read_parquet(f'data/real/{gdf_name}.parquet')

    gdf_geometry      = gdf.geometry.copy()
    cluster_variables = dataset_variables[gdf_name]
    methods           = dataset_methods[gdf_name]
    gdf_attributes    = gdf[cluster_variables].copy()

    task_combinations = []
    for method in ["local", "borders", "random"]:
        c_run_list = get_available_c_runs(gdf_name, method) if method == "local" else c_runs

        for c_run in c_run_list:
            if method == "random":
                available = list(range(1, 501)) #first 500 deleted edges
            else:
                available = get_available_intensities(gdf_name, method, c_run, subfolder="one_component")
            #print(available)
            sampled = select_sample_graphs(available, n_sample=20)
            print(f"{sampled}+{available}")
            for intensity in sampled:
                task_combinations.append((gdf_name, intensity, c_run, method))
    

    results = Parallel(n_jobs=-1)(
        delayed(_process_regionalization_real)(
            gdf_name, missing_edges, c_run, method, gdf_geometry, gdf_attributes, cluster_variables
        )
        for gdf_name,missing_edges, c_run, method in task_combinations
    )

    res_df = pd.concat(results, ignore_index=True)

    output_path = f"results/regionalization/real/extra/reg_{gdf_name}_methods.parquet"
    res_df.to_parquet(output_path, index=False)
    print(f"Saved: {output_path}  |  shape: {res_df.shape}  |  tasks: {len(task_combinations)}\n")